# Decoder trained on the cut treadmill

Trial-averaged error traces of the **cut-treadmill** Ridge decoder, compared between
**Layer 2/3** and **Layer 5** across the RS × OF condition grid.

This notebook **reloads the fitted predictions** produced by the decoder pipeline
(`ridge_decoder_predictions_motor_cut.parquet`, via `run_full_cut.py`) rather than refitting
in-notebook. The pipeline trains and cross-validates on the cut (steady-state) trials, so the
error traces span the steady-state portion of each trial.

Targets: Running Speed (RS), Optic Flow (OF), Depth, and the depth-orthogonal **RS×OF**
product (`rsof_product_stim`). Depth is the RS/OF ratio in log space; RS×OF is the orthogonal
axis ($\log RS + \log OF$). All errors are squared errors in log units.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

from cottage_analysis.summary_analysis import load_project_predictions, load_project_subsets

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

PROJECTS = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
CUT_OFF = 350      # nominal depth (um) splitting Layer 2/3 from Layer 5
COND = "closedloop"  # motor sessions are closed-loop by definition

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording"
}

# Nominal condition grid used to snap each trial to its (RS, OF) cell.
ALL_SPEEDS = np.array([3.8125, 7.0, 15.0, 30.0, 60.0]) / 100      # m/s
ALL_OFS = np.deg2rad(np.array([1.0, 4.0, 16.0, 64.0, 256.0, 1024.0]))  # rad/s

# Decoder targets: (prediction/true column stem, display label)
TARGETS = [
    ("RS_stim", "Running Speed (RS)"),
    ("OF_stim", "Optic Flow (OF)"),
    ("depth", "Depth"),
    ("rsof_product_stim", "RS×OF (depth-orthogonal)"),
]

## Load the fitted predictions

`run_full_cut.py` writes one `ridge_decoder_predictions_motor_cut.parquet` per session, with
`ridge_pred_<target>_closedloop` / `ridge_true_<target>_closedloop` arrays (log units) for
every target including `rsof_product_stim`.

In [ ]:
dfs = []
for project in PROJECTS:
    df = load_project_predictions(project, session_to_exclude=SESSIONS_TO_EXCLUDE.keys(), cut_treadmill=True)
    if not df.empty:
        df["project"] = project
        dfs.append(df)
df_preds = pd.concat(dfs, ignore_index=True)

print(f"{len(df_preds)} trials from {df_preds['session'].nunique()} sessions")
missing = [c for t, _ in TARGETS for c in (f"ridge_pred_{t}_{COND}", f"ridge_true_{t}_{COND}")
           if c not in df_preds.columns]
print("Missing expected columns:", missing or "none")

## Compute per-trial squared-error traces and snap each trial to its (RS, OF) cell

In [ ]:
def snap(value_log, grid):
    """Snap a log-space value to the closest grid point (grid given in linear units)."""
    return min(grid, key=lambda x: abs(np.log(x) - value_log))

records = []
for _, row in df_preds.iterrows():
    rs_true = row.get(f"ridge_true_RS_stim_{COND}")
    of_true = row.get(f"ridge_true_OF_stim_{COND}")
    if rs_true is None or of_true is None:
        continue
    rs_med, of_med = np.nanmedian(rs_true), np.nanmedian(of_true)
    if np.isnan(rs_med) or np.isnan(of_med):
        continue

    rec = {"session": row["session"], "nominal_depth": row["nominal_depth"],
           "expected_RS": snap(rs_med, ALL_SPEEDS), "expected_OF": snap(of_med, ALL_OFS),
           "RS_lin": np.exp(np.asarray(rs_true, dtype=float))}  # true RS in m/s for background

    ok = True
    for col, _ in TARGETS:
        pred = row.get(f"ridge_pred_{col}_{COND}")
        true = row.get(f"ridge_true_{col}_{COND}")
        if pred is None or true is None:
            ok = False
            break
        rec[f"{col}_error"] = (np.asarray(true, dtype=float) - np.asarray(pred, dtype=float)) ** 2
    if ok:
        records.append(rec)

trials_pred = pd.DataFrame(records)
trials_pred["layer"] = np.where(trials_pred["nominal_depth"] <= CUT_OFF, "Layer 2/3", "Layer 5")
TRACE_COLS = [f"{col}_error" for col, _ in TARGETS] + ["RS_lin"]
print(f"{len(trials_pred)} usable trials")

In [ ]:
def stack_mean(arrays, with_sem=False):
    """Truncate a list of 1-D arrays to the shortest, stack, and average (optionally + SEM)."""
    arrs = [np.asarray(a, dtype=float) for a in arrays
            if isinstance(a, (np.ndarray, list, pd.Series)) and len(a) > 0]
    if not arrs:
        return (np.nan, np.nan) if with_sem else np.nan
    m = min(len(a) for a in arrs)
    s = np.stack([a[:m] for a in arrs])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(s, axis=0)
        if with_sem:
            return mean, stats.sem(s, axis=0, nan_policy="omit")
    return mean

# Stage 1: average traces within each session x condition
session_rows = []
for (sess, layer, of, speed), g in trials_pred.groupby(["session", "layer", "expected_OF", "expected_RS"]):
    rd = {"session": sess, "layer": layer, "expected_OF": of, "expected_RS": speed}
    for col in TRACE_COLS:
        rd[f"{col}_mean"] = stack_mean(g[col].values)
    session_rows.append(rd)
df_session = pd.DataFrame(session_rows)

# Stage 2: average session-level traces within each layer x condition (mean +/- SEM)
averaged_rows = []
for (layer, of, speed), g in df_session.groupby(["layer", "expected_OF", "expected_RS"]):
    rd = {"layer": layer, "expected_OF": of, "expected_RS": speed, "n_sessions": g["session"].nunique()}
    for col in TRACE_COLS:
        rd[f"{col}_mean"], rd[f"{col}_sem"] = stack_mean(g[f"{col}_mean"].values, with_sem=True)
    averaged_rows.append(rd)
averaged_df = pd.DataFrame(averaged_rows)

## Decoder R² vs neuron subset size, by layer

Cross-validated decoder R² as a function of the number of neurons used, averaged across
sessions within each layer (mean ± SEM), for all four targets. Each session's largest
(full-population) subset size is excluded. This uses the neuron-subset results
(`ridge_decoder_neuron_subsets_motor_cut.parquet`) from the same cut-treadmill run.

In [ ]:
dfs = []
for project in PROJECTS:
    df = load_project_subsets(project, session_to_exclude=SESSIONS_TO_EXCLUDE.keys(), cut_treadmill=True)
    if not df.empty:
        df["project"] = project
        dfs.append(df)
subsets_all_df = pd.concat(dfs, ignore_index=True)
print("Targets:", subsets_all_df["target"].unique())
print(f"{subsets_all_df['session'].nunique()} sessions")

In [ ]:
layer_colors = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}

# Colour encodes layer; marker shape encodes the decoding target.
target_markers = {"OF_stim": "o", "RS_stim": "s", "depth": "^", "rsof_product_stim": "D"}

# Exclude each session's largest subset size (the full-population point)
max_sizes = subsets_all_df.groupby("session")["subset_size"].transform("max")
filtered_subsets = subsets_all_df[subsets_all_df["subset_size"] < max_sizes]

fig, ax = plt.subplots(figsize=(8, 6))
for col, label in TARGETS[:-1]:
    cond_df = filtered_subsets[(filtered_subsets["target"] == col) & (filtered_subsets["condition"] == COND)]
    marker = target_markers[col]
    for layer, ls in [("Layer 2/3", "-"), ("Layer 5", "--")]:
        if layer == "Layer 2/3":
            ld = cond_df[cond_df.nominal_depth <= CUT_OFF]
        else:
            ld = cond_df[cond_df.nominal_depth > CUT_OFF]
        if ld.empty:
            continue
        color = layer_colors[layer]
        g = ld.groupby("subset_size")["r2_mean"]
        m, s = g.mean(), g.sem()
        ax.plot(m.index, m.values, marker=marker, linestyle=ls, color=color, linewidth=2, alpha=0.5)
        ax.fill_between(m.index, m.values - s.values, m.values + s.values, color=color, alpha=0.10)

ax.set_xscale("log")
ax.set_xlabel("Subset size (# neurons)", fontsize=18)
ax.set_ylabel("Mean $R^2$", fontsize=18)
ax.tick_params(axis="both", which="major", labelsize=18)
ax.set_ylim(0, 1)
ax.grid(True, linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Two-part legend: colour = layer, marker = target
from matplotlib.lines import Line2D
layer_handles = [Line2D([0], [0], color=c, lw=2, label=lyr) for lyr, c in layer_colors.items()]
target_handles = [Line2D([0], [0], color="grey", marker=target_markers[col], linestyle="none",
                         markersize=8, label=label) for col, label in TARGETS[:-1]]
leg1 = ax.legend(handles=layer_handles, frameon=False, bbox_to_anchor=(1.02, 1),
                 loc="upper left", fontsize=14, title="Layer")
ax.add_artist(leg1)
ax.legend(handles=target_handles, frameon=False, bbox_to_anchor=(1.02, 0.6),
          loc="upper left", fontsize=14, title="Target")

plt.tight_layout() 
fig.savefig("decoder_treadmill_cut_bysubsetsize.svg", dpi=600, bbox_inches="tight")


## Decoding error matrices across the RS × OF grid

Mean decoding error in log space (RMSE of the trial-averaged squared-error trace) for each
condition, with running speed on the x-axis and optic flow on the y-axis. One matrix per
target, for each layer (top row Layer 2/3, bottom row Layer 5). The colour scale is shared
across layers within each target.

In [ ]:
# Scalar decoding error (log-space RMSE) per (layer, OF, RS, target),
# from the trial-averaged squared-error traces in averaged_df.
scalar_rows = []
for _, r in averaged_df.iterrows():
    rd = {"layer": r["layer"], "expected_OF": r["expected_OF"], "expected_RS": r["expected_RS"]}
    for col, _ in TARGETS:
        arr = r[f"{col}_error_mean"]
        rd[col] = np.sqrt(np.nanmean(arr)) if isinstance(arr, np.ndarray) else np.nan
    scalar_rows.append(rd)
err_scalar = pd.DataFrame(scalar_rows)

layers = ["Layer 2/3", "Layer 5"]
fig, axes = plt.subplots(len(layers), len(TARGETS), figsize=(5 * len(TARGETS), 8.5))
for c, (col, label) in enumerate(TARGETS):
    vmax = np.nanmax(err_scalar[col].values)
    vmax = vmax if np.isfinite(vmax) else None
    for rr, layer in enumerate(layers):
        ax = axes[rr, c]
        ld = err_scalar[err_scalar["layer"] == layer]
        piv = ld.pivot(index="expected_OF", columns="expected_RS", values=col).sort_index().sort_index(axis=1)
        sns.heatmap(piv, annot=True, fmt=".2f", cmap="magma_r", vmin=0, vmax=vmax,
                    cbar_kws={"label": "log RMSE"}, ax=ax,
                    xticklabels=[f"{rs * 100:.3g}" for rs in piv.columns],
                    yticklabels=[f"{np.rad2deg(of):.0f}" for of in piv.index])
        ax.invert_yaxis()
        ax.set_title(f"{label}\n{layer}", fontsize=11)
        ax.set_xlabel("Running speed (cm/s)")
        ax.set_ylabel("Optic flow (deg/s)")
fig.suptitle("Decoding error (log-space RMSE) across the RS × OF grid", fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

## Decoding geometry: which directions in the (logRS, logOF) plane are read out best?

Running speed (RS), optic flow (OF), depth and RS×OF are **not independent variables** — in log
space they are all directions in the single 2-D plane spanned by `logRS` and `logOF`:

| variable | definition | direction `u = (RS, OF)` | angle θ |
|---|---|---|---|
| RS    | `logRS`           | `(1, 0)`   | 0°   |
| RS×OF | `logRS + logOF`   | `(1, 1)`   | 45°  |
| OF    | `logOF`           | `(0, 1)`   | 90°  |
| depth | `logRS − logOF`   | `(1, −1)`  | 135° |

So "how well is variable X decoded" is really "how well is **direction `u`** decoded". We answer
this for *every* direction at once, from a **single** cross-validated decoder of the plane — the
pipeline already provides it as the held-out `ridge_pred_RS_stim` / `ridge_pred_OF_stim`.

For a direction `u`, with held-out predictions `ŷ` and truths `y` (both 2-D = (logRS, logOF)):

- signal variance along `u` = `uᵀ Σ_sig u`, where `Σ_sig = cov(y)`
- residual variance along `u` = `uᵀ Σ_err u`, where `Σ_err = cov(ŷ − y)`
- **decodability** `R²(u) = 1 − (uᵀ Σ_err u) / (uᵀ Σ_sig u)`

Because every direction comes from the *same* fit, all variables are compared on equal footing —
only the angle changes. We do this per session, then compare layers.

### Step 1 — per-session error and signal covariance

For each session we concatenate the held-out frames, drop NaNs, and form the two 2×2 matrices
`Σ_err` (decoding-error covariance) and `Σ_sig` (signal covariance) in `(logRS, logOF)` space.

In [ ]:
def _concat(g, col):
    """Concatenate a per-trial column of arrays into one 1-D array."""
    arrs = [np.asarray(x, float) for x in g[col] if x is not None]
    return np.concatenate(arrs) if arrs else np.array([])

# session -> (Sigma_err, Sigma_sig, layer)
plane_cov = {}
for sess, g in df_preds.groupby("session"):
    rs_true = _concat(g, f"ridge_true_RS_stim_{COND}")
    rs_pred = _concat(g, f"ridge_pred_RS_stim_{COND}")
    of_true = _concat(g, f"ridge_true_OF_stim_{COND}")
    of_pred = _concat(g, f"ridge_pred_OF_stim_{COND}")
    if min(len(rs_true), len(of_true)) < 10:
        continue
    # keep frames where all four values are finite
    ok = ~(np.isnan(rs_true) | np.isnan(rs_pred) | np.isnan(of_true) | np.isnan(of_pred))
    # rows = (logRS, logOF); Sigma_err from prediction errors, Sigma_sig from the truths
    sigma_err = np.cov(np.vstack([rs_true[ok] - rs_pred[ok], of_true[ok] - of_pred[ok]]))
    sigma_sig = np.cov(np.vstack([rs_true[ok], of_true[ok]]))
    layer = "L2/3" if g["nominal_depth"].iloc[0] <= CUT_OFF else "L5"
    plane_cov[sess] = (sigma_err, sigma_sig, layer)

n_by_layer = pd.Series([v[2] for v in plane_cov.values()]).value_counts().to_dict()
print(f"{len(plane_cov)} sessions: {n_by_layer}")

### Step 2 — angular decodability profile R²(θ)

Sweep the readout direction `u(θ) = (cosθ, sinθ)` from 0° to 180° and evaluate `R²(u)` for every
session. Two complementary views:

- **Left — absolute R²(θ)** (mean ± SEM across sessions). The band is wide because *overall*
  decodability differs a lot between sessions (the whole curve sits higher or lower), not because
  the shape is uncertain.
- **Right — session-centred R²(θ)**: each session's curve minus its own mean over θ, which removes
  that per-session level and isolates the **shape** — which directions are easier *relative to the
  session's own baseline*. The bands here are tight, showing the depth-vs-RS×OF / OF-vs-RS pattern
  is consistent within sessions even though absolute levels vary. This is the visual analogue of the
  *paired* statistical tests below.

In [ ]:
# Direction-resolved R² for a 2x2 (Sigma_err, Sigma_sig) over all angles at once.
angles = np.arange(0, 180)                                   # degrees
U = np.vstack([np.cos(np.deg2rad(angles)), np.sin(np.deg2rad(angles))])  # 2 x nAngles

def r2_profile(sigma_err, sigma_sig):
    # uᵀ Σ u for every column u of U, vectorised
    resid = np.einsum("ij,ik,jk->k", sigma_err, U, U)
    signal = np.einsum("ij,ik,jk->k", sigma_sig, U, U)
    return 1 - resid / signal

# one R²(θ) curve per session, grouped by layer
curves = {"L2/3": [], "L5": []}
for sigma_err, sigma_sig, layer in plane_cov.values():
    curves[layer].append(r2_profile(sigma_err, sigma_sig))
curves = {k: np.vstack(v) for k, v in curves.items()}

def mean_sem(arr):
    return arr.mean(0), arr.std(0, ddof=1) / np.sqrt(arr.shape[0])

MARKS = [(0, "RS", "C1"), (45, "RS×OF", "C3"), (90, "OF", "C0"), (135, "depth", "C2")]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
layer_colors = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}
for layer, color in layer_colors.items():
    lshort = "L2/3" if layer == "Layer 2/3" else "L5"
    arr = curves[lshort]
    # Left: absolute R²(θ), mean ± SEM
    m, se = mean_sem(arr)
    axes[0].plot(angles, m, color=color, lw=2.5, label=f"{layer}")
    axes[0].fill_between(angles, m - se, m + se, color=color, alpha=0.2)
    # Right: session-centred (each curve minus its own mean over θ), mean ± SEM
    centred = arr - arr.mean(1, keepdims=True)
    mc, sec = mean_sem(centred)
    axes[1].plot(angles, mc, color=color, lw=2.5, label=f"{layer}")
    axes[1].fill_between(angles, mc - sec, mc + sec, color=color, alpha=0.2)

ylim = dict(zip(axes, [(0.4, 0.8), (-0.2, 0.1)]))
for ax, ttl, ylab in [(axes[0], "Absolute R²(θ)  (mean ± SEM)", "R²(θ)"),
                      (axes[1], "Session-centred R²(θ)  (mean ± SEM)", "R²(θ) − session mean over θ")]:
    ax.set_ylim(*ylim[ax])
    for angle, name, col in MARKS:
        ax.axvline(angle, 0, 1, color='k', ls=":", lw=1.5)
        ax.text(angle, 1.1, name, color='k', ha="center", va="top", fontsize=15,
                transform=ax.get_xaxis_transform())
    ax.set_xticks([0, 45, 90, 135, 180]); ax.set_xlabel("Readout direction θ (°)")
    ax.set_yticks(ax.get_yticks()[::2])
    ax.set_ylabel(ylab)
    #ax.set_title(ttl, pad=14)
    
    
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0].legend(frameon=False, loc="lower right")
axes[1].axhline(0, color="grey", ls=":")
plt.tight_layout(); plt.show()

# Quantify: inter-session spread of the LEVEL vs consistency of the SHAPE
for layer in ["L2/3", "L5"]:
    arr = curves[layer]
    level_sd = arr.mean(1).std(ddof=1)                 # SD across sessions of the overall level
    d_depth_rsof = arr[:, 135] - arr[:, 45]            # within-session depth − RS×OF
    d_rs_of = arr[:, 0] - arr[:, 90]                   # within-session RS − OF
    print(f"{layer}: across-session level SD={level_sd:.3f} | "
          f"depth−RS×OF {d_depth_rsof.mean():.3f}±{d_depth_rsof.std(ddof=1)/np.sqrt(len(arr)):.3f} (SEM) | "
          f"RS−OF {d_rs_of.mean():.3f}±{d_rs_of.std(ddof=1)/np.sqrt(len(arr)):.3f} (SEM)")

fig.savefig("decoder_treadmill_cut_r2profiles.svg", dpi=600, bbox_inches="tight")

### Step 2b — best decoding direction and what drives it

`R²(θ)` peaks at a **best-decoded direction**. Maximising `R²(u) = 1 − uᵀΣ_err u / uᵀΣ_sig u`
is equivalent to maximising the signal-to-error ratio `uᵀΣ_sig u / uᵀΣ_err u`, whose maximiser is
the **leading generalised eigenvector of (Σ_sig, Σ_err)** (`scipy.linalg.eigh(Σ_sig, Σ_err)`),
with `R²_best = 1 − 1/λ_max`.

To see *why* a direction wins, we decompose it into the two ingredients:
- **signal geometry** `Σ_sig` (how the true logRS / logOF vary in the experiment): its principal
  axis and the OF/RS signal-variance ratio;
- **error geometry** `Σ_err` (the population's decoding-error covariance): its low-error axis, the
  OF/RS error-variance ratio, and the RS–OF error correlation ρ.

In [ ]:
from scipy.linalg import eigh

def _angle(v):
    """Angle (deg, mod 180) of a 2-vector in the (logRS, logOF) plane. 0=RS, 90=OF."""
    return np.degrees(np.arctan2(v[1], v[0])) % 180

dir_rows = []
for sess, (Se, Sy, layer) in plane_cov.items():
    # best direction = leading generalised eigenvector of (Sigma_sig, Sigma_err)
    lam, V = eigh(Sy, Se)            # solves Sy v = lam Se v, eigenvalues ascending
    v_best, lam_best = V[:, -1], lam[-1]
    _, Vs = eigh(Sy)                 # signal principal axis = top eigenvector of Sigma_sig
    _, Ve = eigh(Se)                 # low-error axis = smallest eigenvector of Sigma_err
    dir_rows.append(dict(
        session=sess, layer=layer,
        best_angle=_angle(v_best), R2_best=1 - 1 / lam_best,
        sig_axis=_angle(Vs[:, -1]), low_err_axis=_angle(Ve[:, 0]),
        sig_OF_RS=Sy[1, 1] / Sy[0, 0], err_OF_RS=Se[1, 1] / Se[0, 0],
        sig_corr=Sy[0, 1] / np.sqrt(Sy[0, 0] * Sy[1, 1]),
        err_corr=Se[0, 1] / np.sqrt(Se[0, 0] * Se[1, 1])))
direc = pd.DataFrame(dir_rows)

cols = ["best_angle", "R2_best", "sig_axis", "sig_OF_RS", "sig_corr",
        "low_err_axis", "err_OF_RS", "err_corr"]
print("Per-layer medians (angles in deg; 0=RS, 45=RS×OF, 90=OF, 135=depth):")
print(direc.groupby("layer")[cols].median().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) best decoding direction by layer, with the four named axes marked
ax = axes[0]
sns.boxplot(data=direc, x="layer", y="best_angle", order=["L2/3", "L5"], width=0.4,
            showfliers=False, boxprops=dict(alpha=0.3), ax=ax)
sns.stripplot(data=direc, x="layer", y="best_angle", order=["L2/3", "L5"],
              color="black", alpha=0.6, ax=ax)
for angle, name, c in [(0, "RS", "C1"), (45, "RS×OF", "C3"), (90, "OF", "C0"), (135, "depth", "C2")]:
    ax.axhline(angle, color=c, ls=":", lw=1.5)
    ax.text(1.55, angle, name, color=c, va="center", fontsize=9)
ax.set_ylim(0, 180); ax.set_yticks([0, 45, 90, 135, 180])
ax.set_ylabel("Best decoding direction (°)"); ax.set_xlabel("")
ax.set_title("Best-decoded direction by layer")

# (b) OF/RS variance ratio for signal vs error (why the optimum sits near OF/depth)
ax = axes[1]
melt = direc.melt(id_vars="layer", value_vars=["sig_OF_RS", "err_OF_RS"],
                  var_name="kind", value_name="ratio")
melt["kind"] = melt["kind"].map({"sig_OF_RS": "signal", "err_OF_RS": "error"})
sns.boxplot(data=melt, x="layer", y="ratio", hue="kind", order=["L2/3", "L5"],
            showfliers=False, ax=ax)
ax.axhline(1, color="grey", ls=":")
ax.set_ylabel("OF / RS variance ratio"); ax.set_xlabel(""); ax.legend(frameon=False, title="")
ax.set_title("Variance is OF-dominated (signal more than error)")

for ax in axes:
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

**Interpretation.** The best-decoded direction is where signal variance is large *relative to*
decoding-error variance.

1. **Signal geometry sets the overall tilt toward OF/depth.** Optic flow is sampled over a much
   wider log range than running speed, so the signal covariance has ~9× more variance along OF than
   RS (principal axis ≈ OF) and RS/OF are near-uncorrelated. There is simply much more OF variance to
   explain, so the pure-RS direction is decoded worst and directions with an OF component win.
2. **Error geometry sets the layer-specific rotation.** Decoding error is also OF-heavy, and more so
   in L5 (OF/RS error ratio larger), with a positive RS–OF error correlation (ρ≈0.10) that lowers
   error along the depth (difference) axis. Because L5's error is more OF-concentrated, its optimum
   rotates further from OF toward depth (~155°) than L2/3's (~113°).

*Caveat:* the OF-vs-RS signal asymmetry is partly a **stimulus-sampling** feature (OF spans a wider
range by design), so the generalised-eigen / signal-to-error view used here — not raw signal
magnitude — is the relevant notion of "best decodable direction".

In [ ]:
# reload RF table
rf_path = "/nemo/lab/znamenskiyp/home/shared/projects/ccyp_l5_3d_vision/rf_retinotopy_summary.parquet"
rf_summary = pd.read_parquet(rf_path)
frac_sig_by_session = rf_summary.set_index("session")["frac_sig"]

# Session order that matches the rows of curves[layer]: curves was built by iterating
# plane_cov and appending per layer (cell above), so replay the same iteration here.
sessions_by_layer = {"L2/3": [], "L5": []}
for sess, (_, _, layer) in plane_cov.items():
    sessions_by_layer[layer].append(sess)

# Fraction of significant RF aligned row-for-row to curves[layer]; NaN where a decoder
# session has no entry in the RF table.
frac_sig_by_layer = {
    layer: np.array([frac_sig_by_session.get(s, np.nan) for s in sess_list])
    for layer, sess_list in sessions_by_layer.items()
}
n_sessions = sum(len(v) for v in sessions_by_layer.values())
n_missing = int(sum(np.isnan(v).sum() for v in frac_sig_by_layer.values()))
print(f"RF table: {len(rf_summary)} sessions | decoder: {n_sessions} sessions "
      f"| {n_missing} without a frac_sig match")


In [ ]:
# Plot single sessions, colour-coded by fraction of significant RF
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

cmap = plt.get_cmap("viridis")
missing_color = "lightgray"
all_fracs = np.concatenate([frac_sig_by_layer["L2/3"], frac_sig_by_layer["L5"]])
norm = Normalize(vmin=np.nanmin(all_fracs), vmax=np.nanmax(all_fracs))

fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True)
for layer in ("L2/3", "L5"):
    row = 0 if layer == "L2/3" else 1
    arr = curves[layer]
    fracs = frac_sig_by_layer[layer]
    centred = arr - arr.mean(1, keepdims=True)      # each curve minus its own mean over theta
    for abs_curve, cen_curve, frac in zip(arr, centred, fracs):
        color = cmap(norm(frac)) if np.isfinite(frac) else missing_color
        axes[row, 0].plot(angles, abs_curve, color=color, lw=1, alpha=0.8)
        axes[row, 1].plot(angles, cen_curve, color=color, lw=1, alpha=0.8)
    axes[row, 0].set_title(f"{layer} - absolute R2(theta)  (n={len(arr)})", pad=14)
    axes[row, 1].set_title(f"{layer} - session-centred R2(theta)  (n={len(arr)})", pad=14)

for r in range(2):
    for c in range(2):
        ax = axes[r, c]
        for angle, name, col in MARKS:
            ax.axvline(angle, 0, 0.9, color=col, ls=":", lw=1.5)
            ax.text(angle, 0.99, name, color=col, ha="center", va="top", fontsize=13,
                    transform=ax.get_xaxis_transform())
        ax.set_xticks([0, 45, 90, 135, 180])
        ax.set_xlabel("Readout direction theta (deg)")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0, 0].set_ylabel("R2(theta)"); axes[1, 0].set_ylabel("R2(theta)")
axes[0, 1].set_ylabel("R2(theta) - session mean over theta")
axes[1, 1].set_ylabel("R2(theta) - session mean over theta")
axes[0, 1].axhline(0, color="grey", ls=":"); axes[1, 1].axhline(0, color="grey", ls=":")

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
cbar.set_label("Fraction of significant RF")
plt.show()


In [ ]:
# Plot single sessions, colour-coded by number of significant RF cells
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Number of significant cells aligned row-for-row to curves[layer]; NaN where a decoder
# session has no entry in the RF table.
n_sig_by_session = rf_summary.set_index("session")["n_sig"]
n_sig_by_layer = {
    layer: np.array([n_sig_by_session.get(s, np.nan) for s in sess_list], dtype=float)
    for layer, sess_list in sessions_by_layer.items()
}

cmap = plt.get_cmap("viridis")
missing_color = "lightgray"
all_counts = np.concatenate([n_sig_by_layer["L2/3"], n_sig_by_layer["L5"]])
norm = Normalize(vmin=np.nanmin(all_counts), vmax=np.nanmax(all_counts))

fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True)
for layer in ("L2/3", "L5"):
    row = 0 if layer == "L2/3" else 1
    arr = curves[layer]
    counts = n_sig_by_layer[layer]
    centred = arr - arr.mean(1, keepdims=True)      # each curve minus its own mean over theta
    for abs_curve, cen_curve, count in zip(arr, centred, counts):
        color = cmap(norm(count)) if np.isfinite(count) else missing_color
        axes[row, 0].plot(angles, abs_curve, color=color, lw=1, alpha=0.8)
        axes[row, 1].plot(angles, cen_curve, color=color, lw=1, alpha=0.8)
    axes[row, 0].set_title(f"{layer} - absolute R2(theta)  (n={len(arr)})", pad=14)
    axes[row, 1].set_title(f"{layer} - session-centred R2(theta)  (n={len(arr)})", pad=14)

for r in range(2):
    for c in range(2):
        ax = axes[r, c]
        for angle, name, col in MARKS:
            ax.axvline(angle, 0, 0.9, color=col, ls=":", lw=1.5)
            ax.text(angle, 0.99, name, color=col, ha="center", va="top", fontsize=13,
                    transform=ax.get_xaxis_transform())
        ax.set_xticks([0, 45, 90, 135, 180])
        ax.set_xlabel("Readout direction theta (deg)")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0, 0].set_ylabel("R2(theta)"); axes[1, 0].set_ylabel("R2(theta)")
axes[0, 1].set_ylabel("R2(theta) - session mean over theta")
axes[1, 1].set_ylabel("R2(theta) - session mean over theta")
axes[0, 1].axhline(0, color="grey", ls=":"); axes[1, 1].axhline(0, color="grey", ls=":")

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
cbar.set_label("Number of significant RF cells")
plt.show()


In [ ]:
MIN_FRAC_SIG_RF = 0.25
MIN_N_SIG_RF_CELL = 50

In [ ]:
# Plot single sessions, blue if the session passes BOTH RF criteria, red otherwise
# (grey where the session has no entry in the RF table).
pass_color, fail_color, missing_color = "tab:blue", "tab:red", "lightgray"

pass_by_layer = {
    layer: (frac_sig_by_layer[layer] >= MIN_FRAC_SIG_RF)
    & (n_sig_by_layer[layer] >= MIN_N_SIG_RF_CELL)
    for layer in ("L2/3", "L5")
}

fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True)
for layer in ("L2/3", "L5"):
    row = 0 if layer == "L2/3" else 1
    arr = curves[layer]
    fracs = frac_sig_by_layer[layer]
    counts = n_sig_by_layer[layer]
    passes = pass_by_layer[layer]
    centred = arr - arr.mean(1, keepdims=True)      # each curve minus its own mean over theta
    n_pass = int(np.nansum(passes))
    for abs_curve, cen_curve, frac, count, ok in zip(arr, centred, fracs, counts, passes):
        if not (np.isfinite(frac) and np.isfinite(count)):
            color = missing_color
        else:
            color = pass_color if ok else fail_color
        axes[row, 0].plot(angles, abs_curve, color=color, lw=1, alpha=0.8)
        axes[row, 1].plot(angles, cen_curve, color=color, lw=1, alpha=0.8)
    axes[row, 0].set_title(f"{layer} - absolute R2(theta)  ({n_pass}/{len(arr)} pass)", pad=14)
    axes[row, 1].set_title(f"{layer} - session-centred R2(theta)  ({n_pass}/{len(arr)} pass)", pad=14)

for r in range(2):
    for c in range(2):
        ax = axes[r, c]
        for angle, name, col in MARKS:
            ax.axvline(angle, 0, 0.9, color=col, ls=":", lw=1.5)
            ax.text(angle, 0.99, name, color=col, ha="center", va="top", fontsize=13,
                    transform=ax.get_xaxis_transform())
        ax.set_xticks([0, 45, 90, 135, 180])
        ax.set_xlabel("Readout direction theta (deg)")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0, 0].set_ylabel("R2(theta)"); axes[1, 0].set_ylabel("R2(theta)")
axes[0, 1].set_ylabel("R2(theta) - session mean over theta")
axes[1, 1].set_ylabel("R2(theta) - session mean over theta")
axes[0, 1].axhline(0, color="grey", ls=":"); axes[1, 1].axhline(0, color="grey", ls=":")

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color=pass_color, lw=2,
           label=f"pass (frac_sig >= {MIN_FRAC_SIG_RF} & n_sig >= {MIN_N_SIG_RF_CELL})"),
    Line2D([0], [0], color=fail_color, lw=2, label="fail"),
    Line2D([0], [0], color=missing_color, lw=2, label="no RF entry"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.show()


### Diagnostic — sessions where RS decodes better than OF

Some single-session curves are U-shaped (R²(θ) higher near RS than near OF), which is unexpected.
This cell flags those sessions and checks whether they show any **OF sampling / coverage pathology**
that could explain it — number of trials, number of distinct OF and RS conditions, and the sampled
variance/range of log-OF. If the flagged sessions have the *same* coverage and OF variance as the
rest, the U-shape reflects genuine session/layer variability rather than a data-processing problem.

In [ ]:
def _snap(v, grid):
    return min(grid, key=lambda x: abs(np.log(x) - v))

diag_rows = []
for sess, g in df_preds.groupby("session"):
    rs_true, rs_pred = _concat(g, f"ridge_true_RS_stim_{COND}"), _concat(g, f"ridge_pred_RS_stim_{COND}")
    of_true, of_pred = _concat(g, f"ridge_true_OF_stim_{COND}"), _concat(g, f"ridge_pred_OF_stim_{COND}")
    if min(len(rs_true), len(of_true)) < 10:
        continue
    ok = ~(np.isnan(rs_true) | np.isnan(rs_pred) | np.isnan(of_true) | np.isnan(of_pred))
    Se = np.cov(np.vstack([rs_true[ok] - rs_pred[ok], of_true[ok] - of_pred[ok]]))
    Sy = np.cov(np.vstack([rs_true[ok], of_true[ok]]))
    # number of distinct OF / RS conditions actually sampled (snapped to the nominal grid)
    of_c, rs_c = set(), set()
    for _, r in g.iterrows():
        a, b = r.get(f"ridge_true_RS_stim_{COND}"), r.get(f"ridge_true_OF_stim_{COND}")
        if a is None or b is None:
            continue
        rm, om = np.nanmedian(a), np.nanmedian(b)
        if np.isnan(rm) or np.isnan(om):
            continue
        rs_c.add(_snap(rm, ALL_SPEEDS)); of_c.add(_snap(om, ALL_OFS))
    diag_rows.append(dict(
        session=sess, layer="L2/3" if g["nominal_depth"].iloc[0] <= CUT_OFF else "L5",
        n_trials=len(g), n_OF_cond=len(of_c), n_RS_cond=len(rs_c),
        r2_RS=1 - Se[0, 0] / Sy[0, 0], r2_OF=1 - Se[1, 1] / Sy[1, 1],
        var_logOF=Sy[1, 1], var_logRS=Sy[0, 0], OF_range=np.ptp(of_true[ok])))
diag = pd.DataFrame(diag_rows)
diag["RS_minus_OF"] = diag["r2_RS"] - diag["r2_OF"]

flagged = diag[diag["r2_RS"] > diag["r2_OF"]].sort_values("RS_minus_OF", ascending=False)
print(f"{len(flagged)}/{len(diag)} sessions decode RS better than OF (U-shaped R²(θ)):\n")
print(flagged[["session", "layer", "n_trials", "n_OF_cond", "n_RS_cond",
               "r2_RS", "r2_OF", "var_logOF", "OF_range"]].round(3).to_string(index=False))

# coverage / OF-variance: flagged (RS>OF) vs the rest — should match if there is no pathology
print("\nMedian OF coverage & variance — flagged (RS>OF) vs rest:")
grp = diag.assign(group=np.where(diag["r2_RS"] > diag["r2_OF"], "RS>OF", "OF>=RS"))
print(grp.groupby("group")[["n_OF_cond", "n_RS_cond", "var_logOF", "OF_range", "n_trials"]].median().round(2).to_string())
print("\nLayer breakdown of flagged sessions:", flagged["layer"].value_counts().to_dict())

### Step 3 — statistical tests at the four named directions

We test the two scientifically meaningful contrasts, each within layer and then **between**
layers:

1. **depth vs RS×OF** (the two diagonals) — the original question.
2. **RS vs OF** (the two cardinals) — the reverse question.

For each session and each contrast we take the paired difference Δ (e.g. `R²(depth) − R²(RS×OF)`).
- *Within a layer*: a paired **Wilcoxon** signed-rank test asks whether the two directions differ.
- *Between layers*: the **interaction** — is Δ larger in one layer? — tested with **Mann–Whitney**
  and a label-shuffling **permutation** test on Δ.


In [ ]:
def proj_var(S, u):
    u = np.asarray(u, float)
    return float(u @ S @ u)

# per-session R² along the four named directions, from the same decoder
AXES = {"RS": (1, 0), "OF": (0, 1), "depth": (1, -1), "rsof": (1, 1)}
rows = []
for sess, (Se, Sy, layer) in plane_cov.items():
    row = {"session": sess, "layer": layer}
    for name, u in AXES.items():
        row[name] = 1 - proj_var(Se, u) / proj_var(Sy, u)
    rows.append(row)
geom = pd.DataFrame(rows)
geom["d_depth_rsof"] = geom["depth"] - geom["rsof"]   # contrast 1
geom["d_RS_OF"]      = geom["RS"]    - geom["OF"]      # contrast 2

def test_contrast(a_col, b_col, delta_col, label):
    print(f"\n{label}:")
    for layer in ["L2/3", "L5"]:
        ld = geom[geom.layer == layer]
        _, p = stats.wilcoxon(ld[a_col], ld[b_col])
        print(f"  {layer} (n={len(ld)}): median {a_col}={ld[a_col].median():.3f}, "
              f"{b_col}={ld[b_col].median():.3f}, Δ={ld[delta_col].median():.3f}, Wilcoxon p={p:.4g}")
    dL = geom[geom.layer == "L2/3"][delta_col].values
    d5 = geom[geom.layer == "L5"][delta_col].values
    _, p_mw = stats.mannwhitneyu(dL, d5)
    p_perm = stats.permutation_test((dL, d5), lambda a, b: np.median(a) - np.median(b),
                                    permutation_type="independent", n_resamples=10000,
                                    random_state=0).pvalue
    print(f"  interaction (Δ differs by layer): Mann-Whitney p={p_mw:.4g}, permutation p={p_perm:.4g}")
    return p_mw

p_depth_rsof = test_contrast("depth", "rsof", "d_depth_rsof", "depth vs RS×OF")
p_rs_of      = test_contrast("RS", "OF", "d_RS_OF", "RS vs OF")

In [ ]:
# Plot of the relevant statistical tests: per-session contrast Δ by layer.
# Each panel shows one contrast; within-layer Wilcoxon p is annotated above each layer,
# and the between-layer interaction p (Mann-Whitney) is in the title.
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
panels = [("d_depth_rsof", "depth", "rsof", "Δ = R²(depth) − R²(RS×OF)", p_depth_rsof),
          ("d_RS_OF",       "RS",    "OF",   "Δ = R²(RS) − R²(OF)",       p_rs_of)]
for ax, (col, a, b, title, p_inter) in zip(axes, panels):
    sns.boxplot(data=geom, x="layer", y=col, order=["L2/3", "L5"], width=0.5,
                showfliers=False, boxprops=dict(alpha=0.4), ax=ax)
    sns.stripplot(data=geom, x="layer", y=col, order=["L2/3", "L5"], color="black", alpha=0.6, ax=ax)
    ax.axhline(0, color="grey", ls=":")
    # within-layer paired Wilcoxon p above each layer's points
    ytop = geom[col].max()
    for x, layer in enumerate(["L2/3", "L5"]):
        ld = geom[geom.layer == layer]
        _, p = stats.wilcoxon(ld[a], ld[b])
        ax.text(x, ytop * 0.9, f"p={p:.3f}", ha="center", va="top", fontsize=9)
    ax.set_title(f"{title}\ninteraction p = {p_inter:.3g}")
    ax.set_xlabel(""); ax.set_ylabel("Δ R²")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)



### Step 4 — effect size

The absolute ΔR² is small, but what matters is whether it is **consistent across sessions**. We
report standardised / distribution-free effect sizes for the paired (within-layer) contrasts:

- **Cohen's dₙ** (paired) = mean(Δ) / SD(Δ): the within-session difference in units of its own
  spread (≈0.2 small, 0.5 medium, 0.8 large).
- **rank-biserial r** (the effect size matching the Wilcoxon test): (R⁺ − R⁻)/ΣR, range [−1, 1].
- **common-language effect size** = fraction of sessions where the first variable wins.
- **relative %** = median Δ as a fraction of the two variables' mean R² (how big the gap is in
  proportional terms).

In [ ]:
from scipy.stats import rankdata

def effect_sizes(a, b):
    """Paired effect sizes for a vs b (per-session arrays)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    dz = d.mean() / d.std(ddof=1)
    nz = d[d != 0]
    r = rankdata(np.abs(nz))
    Rpos, Rneg = r[nz > 0].sum(), r[nz < 0].sum()
    rank_biserial = (Rpos - Rneg) / (Rpos + Rneg)
    cles = np.mean(a.values > b.values)
    rel = np.median(d) / np.mean([a.median(), b.median()])
    return dict(n=len(d), mean_delta=d.mean(), median_delta=np.median(d),
                cohen_dz=dz, rank_biserial=rank_biserial, P_first_wins=cles, relative_pct=100 * rel)

rows = []
for first, second, name in [("depth", "rsof", "depth vs RS×OF"), ("RS", "OF", "RS vs OF")]:
    for layer in ["L2/3", "L5"]:
        ld = geom[geom.layer == layer]
        es = effect_sizes(ld[first], ld[second]); es.update(contrast=name, layer=layer)
        rows.append(es)
eff = pd.DataFrame(rows)[["contrast", "layer", "n", "median_delta", "cohen_dz",
                          "rank_biserial", "P_first_wins", "relative_pct"]]
print(eff.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## Per-neuron directional variance on the trial averages (Option A)

The population analysis above asks *which readout direction of the `(logRS, logOF)` plane is
decoded best*. The single-neuron dual asks *which direction each neuron encodes*. For every
neuron we take its **trial-averaged** response (one scalar per trial) and the trial's stimulus
coordinate `(logRS, logOF)`, and compute the fraction of the response variance explained by the
stimulus projection onto each named direction — a model-free 1-D variance decomposition (η²):

| variable | direction `u = (RS, OF)` | angle θ |
|---|---|---|
| RS    | `(1, 0)`   | 0°   |
| RS×OF | `(1, 1)`   | 45°  |
| OF    | `(0, 1)`   | 90°  |
| depth | `(1, −1)`  | 135° |

Because η² is a *fraction of each neuron's own response variance*, the four directions are
comparable within a neuron despite OF being sampled over a wider log range than RS (the
stimulus-sampling caveat from the population analysis). We then compare the paired
per-neuron contrasts **depth − RS×OF** and **RS − OF** across layers, mirroring the population
tests in Step 3.


In [ ]:
# ---------------------------------------------------------------------------
# Per-neuron directional variance on the trial averages
#
# For every neuron we take its trial-AVERAGED response (one scalar per trial) and
# the trial's stimulus coordinate (logRS, logOF), then compute:
#   Option A — η²(u): fraction of the response variance explained by the stimulus
#              projection onto each named direction (model-free 1-D decomposition).
#   Option B — a 2x2 "structure tensor" M = mean over the tuning grid of ∇f ∇fᵀ,
#              so uᵀMu is how much the response changes as you move along u. The
#              gradient is measured per FRACTION OF SAMPLED RANGE (each axis rescaled so
#              its sampled extent = 1), matching RS and OF on "response change across the
#              range probed" -- consistent with Option A's η². Stored as (m_rr, m_ro, m_oo).
#   RS = (1,0)   RS×OF = (1,1)   OF = (0,1)   depth = (1,-1)   [same convention as R²(θ)]
#
# The trial averages come from the shared cache (v1_depth_map.figure_utils.trial_averages),
# built once from load_session and reused, instead of reloading every trials_df here.
# ---------------------------------------------------------------------------
from pathlib import Path
from v1_depth_map.figure_utils import trial_averages as ta

AXES = {"RS": (1, 0), "RSxOF": (1, 1), "OF": (0, 1), "depth": (1, -1)}
DV_CACHE = Path("directional_variance_trial_average_v3.pickle")

# Grid centres in the same log units as the trial averages below
RS_CENTERS = np.sort(np.log(ALL_SPEEDS))            # logRS (m/s)
OF_CENTERS = np.sort(np.log(np.rad2deg(ALL_OFS)))   # log optic flow (deg/s)


def eta2_along(C, r, u, nbins=6):
    """Option A: fraction of r's variance explained by the stimulus projection onto u."""
    u = np.asarray(u, float); u = u / np.linalg.norm(u)
    p = C @ u
    edges = np.quantile(p, np.linspace(0, 1, nbins + 1))
    b = np.clip(np.digitize(p, edges[1:-1]), 0, nbins - 1)
    grand = r.mean()
    ss_tot = np.sum((r - grand) ** 2)
    if ss_tot == 0:
        return np.nan
    ss_bet = sum(m.sum() * (r[m].mean() - grand) ** 2
                 for k in range(nbins) if (m := (b == k)).any())
    return ss_bet / ss_tot


def tuning_matrices(logrs, logof, dff):
    """Bin trial averages onto the (OF x RS) grid -> tuning[n_of, n_rs, n_neurons] (NaN if empty)."""
    i_rs = np.argmin(np.abs(logrs[:, None] - RS_CENTERS[None, :]), axis=1)
    i_of = np.argmin(np.abs(logof[:, None] - OF_CENTERS[None, :]), axis=1)
    n_of, n_rs, N = len(OF_CENTERS), len(RS_CENTERS), dff.shape[1]
    ssum = np.zeros((n_of, n_rs, N)); cnt = np.zeros((n_of, n_rs))
    np.add.at(ssum, (i_of, i_rs), dff)
    np.add.at(cnt, (i_of, i_rs), 1)
    with np.errstate(invalid="ignore"):
        return ssum / cnt[..., None]


def structure_tensor(tuning):
    """Option B: (m_rr, m_ro, m_oo) of M = mean over sampled cells of ∇f ∇fᵀ.
    Gradient is measured per FRACTION OF SAMPLED RANGE: each axis is rescaled so its
    sampled extent spans [0, 1], so RS and OF are matched on "response change across the
    range probed" regardless of log spacing or step count (assumption-free counterpart of
    per-grid-step; consistent with Option A's η²)."""
    rs_n = (RS_CENTERS - RS_CENTERS[0]) / (RS_CENTERS[-1] - RS_CENTERS[0])
    of_n = (OF_CENTERS - OF_CENTERS[0]) / (OF_CENTERS[-1] - OF_CENTERS[0])
    gof, grs = np.gradient(tuning, of_n, rs_n)              # axis0=OF, axis1=RS; per fraction of range
    G = np.stack([grs, gof], axis=-1)
    v = np.all(np.isfinite(G), axis=-1)
    if v.sum() < 3:
        return np.nan, np.nan, np.nan
    Gv = G[v]
    M = Gv.T @ Gv / len(Gv)
    return M[0, 0], M[0, 1], M[1, 1]


dv_df = None
if DV_CACHE.exists():
    dv_df = pd.read_pickle(DV_CACHE)
    if "m_rr" not in dv_df.columns:
        print("Cache lacks structure-tensor columns -> recomputing")
        dv_df = None
    else:
        print(f"Loaded cached directional variance: {len(dv_df)} neurons")

if dv_df is None:
    ta_df = ta.load_trial_averages()   # one row per session; builds on first call
    keep = set(df_preds["session"].unique())
    rows = []
    for _, s in ta_df.iterrows():
        if s["session"] not in keep:
            continue
        logrs = np.log(s["rs"])                     # rs stored in m/s
        logof = np.log(np.degrees(s["of"]))         # of stored in rad/s -> log-deg, as in the fits
        dff = s["dff"].astype(float)                # [n_trials x n_neurons]
        C = np.column_stack([logrs, logof])
        tuning = tuning_matrices(logrs, logof, dff)
        for n in range(dff.shape[1]):
            rec = {"project": s["project"], "session": s["session"], "roi": n,
                   "nominal_depth": s["nominal_depth"], "layer": s["layer"]}
            for name, u in AXES.items():
                rec[name] = eta2_along(C, dff[:, n], u)
            rec["m_rr"], rec["m_ro"], rec["m_oo"] = structure_tensor(tuning[:, :, n])
            rows.append(rec)
        print(f"  {s['session']} ({s['layer']}): {dff.shape[1]} neurons, {len(logrs)} trials")
    dv_df = pd.DataFrame(rows)
    dv_df.to_pickle(DV_CACHE)
    print(f"\nComputed directional variance for {len(dv_df)} neurons; cached to {DV_CACHE}")

dv_df["d_RS_OF"] = dv_df["RS"] - dv_df["OF"]
dv_df["d_depth_rsof"] = dv_df["depth"] - dv_df["RSxOF"]


In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
panels = [("d_depth_rsof", "depth", "RSxOF", "Δ = η²(depth) − η²(RS×OF)"),
          ("d_RS_OF",       "RS",    "OF",    "Δ = η²(RS) − η²(OF)")]
for ax, (col, a, b, title) in zip(axes, panels):
    d = dv_df.dropna(subset=[a, b])
    sns.boxplot(data=d, x="layer", y=col, order=["L2/3", "L5"], width=0.5,
                showfliers=False, boxprops=dict(alpha=0.4), ax=ax)
    ax.axhline(0, color="grey", ls=":")
    ytop = d[col].quantile(0.98) * 1.1
    ax.set_ylim(-ytop, ytop)
    for x, layer in enumerate(["L2/3", "L5"]):
        ld = d[d.layer == layer]
        _, p = stats.wilcoxon(ld[a], ld[b])   # paired across neurons within the layer
        ax.text(x, ytop * 0.8, f"n={len(ld)}\np={p:.1e}", ha="center", va="bottom", fontsize=9)
    dL, d5 = d[d.layer == "L2/3"][col], d[d.layer == "L5"][col]
    _, p_i = stats.mannwhitneyu(dL, d5)       # between-layer interaction
    ax.set_title(f"{title}\ninteraction (L2/3 vs L5) p = {p_i:.2e}")
    ax.set_xlabel(""); ax.set_ylabel("Δ η²")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

# Median directional variance (η²) per direction and layer
print(dv_df.groupby("layer")[list(AXES)].median().round(3).to_string())


### Same contrasts, aggregated by session first

Each neuron's η² is first averaged within a session, then the session means are compared across
layers (n = sessions per layer, paired Wilcoxon within layer, Mann–Whitney between layers). This
matches the session-level unit of the population `R²(θ)` analysis above and controls for
sessions contributing very different neuron counts.


In [ ]:
from scipy import stats

# aggregate neurons -> one value per session (mean η² per direction)
sess_agg = dv_df.groupby(["project", "session", "layer"], as_index=False)[list(AXES)].mean()
sess_agg["d_RS_OF"] = sess_agg["RS"] - sess_agg["OF"]
sess_agg["d_depth_rsof"] = sess_agg["depth"] - sess_agg["RSxOF"]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
panels = [("d_depth_rsof", "depth", "RSxOF", "Δ = η²(depth) − η²(RS×OF)"),
          ("d_RS_OF",       "RS",    "OF",    "Δ = η²(RS) − η²(OF)")]
for ax, (col, a, b, title) in zip(axes, panels):
    d = sess_agg.dropna(subset=[a, b])
    sns.boxplot(data=d, x="layer", y=col, order=["L2/3", "L5"], width=0.5,
                showfliers=False, boxprops=dict(alpha=0.4), ax=ax)
    sns.stripplot(data=d, x="layer", y=col, order=["L2/3", "L5"], color="k",
                  size=3, alpha=0.5, ax=ax)
    ax.axhline(0, color="grey", ls=":")
    ytop = d[col].abs().quantile(0.98) * 1.2
    ax.set_ylim(-ytop, ytop)
    for x, layer in enumerate(["L2/3", "L5"]):
        ld = d[d.layer == layer]
        _, pp = stats.wilcoxon(ld[a], ld[b])   # paired across sessions within the layer
        ax.text(x, ytop * 0.8, f"n={len(ld)}\np={pp:.1e}", ha="center", va="bottom", fontsize=9)
    dL, d5 = d[d.layer == "L2/3"][col], d[d.layer == "L5"][col]
    _, p_i = stats.mannwhitneyu(dL, d5)        # between-layer interaction
    ax.set_title(f"{title}\ninteraction (L2/3 vs L5) p = {p_i:.2f}")
    ax.set_xlabel(""); ax.set_ylabel("Δ η²")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

# Median session-level η² per direction and layer
print(sess_agg.groupby("layer")[list(AXES)].median().round(3).to_string())


## Continuous encoding profile and preferred direction (Option B)

Option A resolves only the four named directions. Option B gives each neuron a **continuous**
profile `V(θ) = uᵀ M u`, where `M = ⟨∇f ∇fᵀ⟩` is the mean outer product of the tuning gradient
over the sampled `(logRS, logOF)` grid (computed in the cell above and stored as `m_rr, m_ro,
m_oo`). `V(θ)` is how much the response changes as you move the stimulus along direction `θ`.

The gradient is measured **per fraction of the sampled range**: each axis is rescaled so its
sampled extent spans `[0, 1]` before differentiating. This matters because the RS and OF grids
cover very different stimulus ranges (RS ~16×, OF ~1024×) at different log resolutions — a
per-unit-log gradient would mechanically inflate the finely-sampled RS axis and make every neuron
look RS-preferring. Range-normalising matches the two axes on *"how much the response varies
across the range actually probed"*, which is exactly what Option A's η² measures (and the analog
of the population `R²(θ)` normalising by `Σ_sig`). So Option B is the continuous, differentiable
counterpart of Option A.

- **Left** — `V(θ)` divided by each neuron's own mean over θ (shape only, since absolute
  sensitivity varies enormously across cells), averaged within layer. Peaks mark the direction
  each layer encodes most steeply.
- **Right** — distribution of each neuron's **preferred encoding axis** (leading eigenvector of
  `M`), the single-neuron analog of the population `best_angle` in Step 2b. `anisotropy`
  (`1 − λ_min/λ_max`, printed per layer) says how directional the tuning is (0 = isotropic,
  1 = purely 1-D); the histogram is restricted to `anisotropy > ANISO_THR` since the preferred
  angle is meaningless for near-isotropic cells.


In [ ]:
# Option B — continuous encoding profile V(θ) = uᵀMu and preferred direction per neuron
a, b, c = dv_df["m_rr"].values, dv_df["m_ro"].values, dv_df["m_oo"].values

# preferred (max-variance) encoding axis and anisotropy from each 2x2 tensor
dv_df["pref_angle"] = np.degrees(0.5 * np.arctan2(2 * b, a - c)) % 180
disc = np.sqrt(((a - c) / 2) ** 2 + b ** 2)
lmax, lmin = (a + c) / 2 + disc, (a + c) / 2 - disc
with np.errstate(invalid="ignore", divide="ignore"):
    dv_df["anisotropy"] = 1 - lmin / lmax          # 0 = isotropic, 1 = purely 1-D

# V(θ) = uᵀMu for every neuron, u = (cosθ, sinθ) in (RS, OF)
angB = np.arange(180); u = np.deg2rad(angB); cu, su = np.cos(u), np.sin(u)
Vt = a[:, None] * cu ** 2 + 2 * b[:, None] * cu * su + c[:, None] * su ** 2
ok = np.isfinite(Vt).all(1) & (Vt.mean(1) > 0)
ANISO_THR = 0.5   # only count the preferred angle of reasonably directional neurons

MARKS = [(0, "RS", "C1"), (45, "RS×OF", "C3"), (90, "OF", "C0"), (135, "depth", "C2")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for layer, color in [("L2/3", "#1a5276"), ("L5", "#b03a2e")]:
    sel = ok & (dv_df["layer"].values == layer)
    shape = Vt[sel] / Vt[sel].mean(1, keepdims=True)   # per-neuron shape (level removed)
    m = shape.mean(0); se = shape.std(0, ddof=1) / np.sqrt(len(shape))
    axes[0].plot(angB, m, color=color, lw=2.5, label=f"{layer} (n={sel.sum()})")
    axes[0].fill_between(angB, m - se, m + se, color=color, alpha=0.2)
    hsel = sel & (dv_df["anisotropy"].values > ANISO_THR)
    axes[1].hist(dv_df["pref_angle"].values[hsel], bins=np.arange(0, 181, 10),
                 color=color, alpha=0.5, density=True, label=f"{layer} (n={hsel.sum()})")
axes[0].set_ylabel("V(θ) / mean$_θ$ V \n(encoding sensitivity)")
axes[0].set_title("Directional encoding\nprofile (per-neuron mean ± SEM)")
axes[1].set_ylabel("Fraction of neurons")
axes[1].set_title(f"Preferred encoding direction\n(anisotropy > {ANISO_THR})")
for ax in axes:
    for angle, name, col in MARKS:
        ax.axvline(angle,0,0.9, color=col, ls=":", lw=1.5)
    ax.set_xticks([0, 45, 90, 135, 180]); ax.set_xlabel("Direction θ (°)")
    ax.legend(frameon=False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for angle, name, col in MARKS:
    axes[0].text(angle, 0.9, name, color=col, ha="center", va="bottom", fontsize=11,
                 transform=axes[0].get_xaxis_transform())
plt.tight_layout(); plt.show()

print(dv_df.groupby("layer")[["pref_angle", "anisotropy"]].median().round(3).to_string())


### Option B, aggregated by session first

The per-neuron structure tensors are averaged within a session (mean of `M = ⟨∇f∇fᵀ⟩` over
the session's neurons), then `V(θ)`, preferred axis and anisotropy are derived from each
session-mean tensor (n = sessions per layer). Note that averaging tensors before taking the
anisotropy measures the *net* directional bias of the session's population — it is lower than a
typical single neuron's anisotropy when neurons within a session prefer different axes.


In [ ]:
# Option B aggregated by session first: average the structure tensor per session
sess_M = dv_df.groupby(["project", "session", "layer"], as_index=False)[["m_rr", "m_ro", "m_oo"]].mean()
a, b, c = sess_M["m_rr"].values, sess_M["m_ro"].values, sess_M["m_oo"].values

sess_M["pref_angle"] = np.degrees(0.5 * np.arctan2(2 * b, a - c)) % 180
disc = np.sqrt(((a - c) / 2) ** 2 + b ** 2)
lmax, lmin = (a + c) / 2 + disc, (a + c) / 2 - disc
with np.errstate(invalid="ignore", divide="ignore"):
    sess_M["anisotropy"] = 1 - lmin / lmax

angB = np.arange(180); u = np.deg2rad(angB); cu, su = np.cos(u), np.sin(u)
Vt = a[:, None] * cu ** 2 + 2 * b[:, None] * cu * su + c[:, None] * su ** 2
ok = np.isfinite(Vt).all(1) & (Vt.mean(1) > 0)

MARKS = [(0, "RS", "C1"), (45, "RS×OF", "C3"), (90, "OF", "C0"), (135, "depth", "C2")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for layer, color in [("L2/3", "#1a5276"), ("L5", "#b03a2e")]:
    sel = ok & (sess_M["layer"].values == layer)
    shape = Vt[sel] / Vt[sel].mean(1, keepdims=True)   # per-session shape (level removed)
    m = shape.mean(0); se = shape.std(0, ddof=1) / np.sqrt(len(shape))
    axes[0].plot(angB, m, color=color, lw=2.5, label=f"{layer} (n={sel.sum()})")
    axes[0].fill_between(angB, m - se, m + se, color=color, alpha=0.2)
    axes[1].hist(sess_M["pref_angle"].values[sel], bins=np.arange(0, 181, 15),
                 color=color, alpha=0.5, density=True, label=layer)
axes[0].set_ylabel("V(θ) / mean$_θ$ V \n(encoding sensitivity)")
axes[0].set_title("Directional encoding profile\n(per-session mean ± SEM)")
axes[1].set_ylabel("Fraction of sessions")
axes[1].set_title("Preferred encoding direction")
for ax in axes:
    for angle, name, col in MARKS:
        ax.axvline(angle, 0, 0.9, color=col, ls=":", lw=1.5)
    ax.set_xticks([0, 45, 90, 135, 180]); ax.set_xlabel("Direction θ (°)")
    ax.legend(frameon=False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for angle, name, col in MARKS:
    axes[0].text(angle, 0.9, name, color=col, ha="center", va="bottom", fontsize=11,
                 transform=axes[0].get_xaxis_transform())
plt.tight_layout(); plt.show()

print(sess_M.groupby("layer")[["pref_angle", "anisotropy"]].median().round(3).to_string())


In [ ]:
# Per-session V(θ): one curve per session, layers in separate panels
sess_M = dv_df.groupby(["project", "session", "layer"], as_index=False)[["m_rr", "m_ro", "m_oo"]].mean()
a, b, c = sess_M["m_rr"].values, sess_M["m_ro"].values, sess_M["m_oo"].values

angB = np.arange(180); u = np.deg2rad(angB); cu, su = np.cos(u), np.sin(u)
Vt = a[:, None] * cu ** 2 + 2 * b[:, None] * cu * su + c[:, None] * su ** 2
Vt = Vt / Vt.mean(1, keepdims=True)          # per-session shape (level removed)
ok = np.isfinite(Vt).all(1)

MARKS = [(0, "RS", "C1"), (45, "RS×OF", "C3"), (90, "OF", "C0"), (135, "depth", "C2")]
fig, axes = plt.subplots(1, 2, figsize=(8, 4.5), sharey=True)
for ax, (layer, color) in zip(axes, [("L2/3", "#1a5276"), ("L5", "#b03a2e")]):
    sel = ok & (sess_M["layer"].values == layer)
    for row in Vt[sel]:
        ax.plot(angB, row, color=color, lw=1, alpha=0.35)
    ax.plot(angB, Vt[sel].mean(0), color="k", lw=2.5, label="mean")
    ax.set_title(f"{layer} (n={sel.sum()})")
    for angle, name, col in MARKS:
        ax.axvline(angle, 0, 0.9, color=col, ls=":", lw=1.5)
    ax.set_xticks([0, 45, 90, 135, 180]); ax.set_xlabel("Direction θ (°)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0].set_ylabel("V(θ) / mean$_θ$ V \n(encoding sensitivity)")
axes[0].legend(frameon=False, loc="upper center")
for angle, name, col in MARKS:
    axes[0].text(angle, 0.9, name, color=col, ha="center", va="bottom", fontsize=11,
                 transform=axes[0].get_xaxis_transform())
plt.suptitle("Per-session directional encoding profiles")
plt.tight_layout(); plt.show()
